# Memory from Scratch

**What you will build:** a small memory system that lets an assistant hold a long conversation without overflowing the context window. You already know the core trick (the list *is* the memory); now we manage that list. This is what frameworks call a *checkpointer* (you'll meet the built-in version in LangGraph, 2.3).

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/06_memory_from_scratch.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run once.
!pip install -q openai

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def ask(system, user, temperature=0):
    """One-shot helper: system + user in, text out."""
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

print("Ready. Model:", MODEL)

## The core mechanic (recap) and its problem

From 0.1: memory is just re-sending previous messages. So the naive memory is "append everything forever":

```
turn 1:  [system, u1, a1]
turn 2:  [system, u1, a1, u2, a2]
turn 3:  [system, u1, a1, u2, a2, u3, a3]   ... grows without bound
```

Two problems appear as it grows: you eventually exceed the **context window**, and every turn gets **slower and more expensive** (you pay for the whole history every time). So real memory is about *choosing what to keep*.

## Strategy 1 — Sliding window (keep the last N turns)

The simplest bounded memory: always keep the system prompt, but only the most recent few exchanges. Cheap and predictable; the cost is that older details fall off the edge.

In [ ]:
class WindowMemory:
    def __init__(self, system, keep_turns=3):
        self.system = system
        self.keep = keep_turns          # number of recent user/assistant PAIRS to keep
        self.history = []               # list of {'role','content'}

    def add(self, role, content):
        self.history.append({"role": role, "content": content})

    def messages(self):
        recent = self.history[-self.keep * 2:]           # last N pairs
        return [{"role": "system", "content": self.system}] + recent

    def say(self, user_text):
        self.add("user", user_text)
        r = client.chat.completions.create(model=MODEL, messages=self.messages(), temperature=0)
        reply = r.choices[0].message.content
        self.add("assistant", reply)
        return reply

m = WindowMemory("You are a concise assistant.", keep_turns=2)
print(m.say("My favourite colour is teal."))
print(m.say("I have a dog named Pixel."))
print(m.say("I live in Bilbao."))
print(m.say("What colour did I mention?"))   # may have fallen out of the 2-turn window!

Run it: with `keep_turns=2`, the colour from the first turn has scrolled out of the window, so the model can't recall it. That's the trade-off in its rawest form. To keep old facts *without* keeping every word, we summarize.

## Strategy 2 — Summary memory (compress the old, keep the recent)

Keep a rolling **summary** of everything older, plus the last few verbatim turns. Older facts survive in compressed form; the context stays bounded. This is the strategy most production assistants use.

In [ ]:
class SummaryMemory:
    def __init__(self, system, keep_turns=2):
        self.system = system
        self.keep = keep_turns
        self.summary = ""              # rolling summary of older turns
        self.recent = []               # verbatim recent messages

    def _summarize(self, old_messages):
        text = "\n".join(f"{m['role']}: {m['content']}" for m in old_messages)
        return ask("Update the running summary. Keep all facts about the user. Be brief.",
                   f"Existing summary:\n{self.summary}\n\nNew turns:\n{text}")

    def messages(self):
        sys = self.system + (f"\n\nConversation summary so far:\n{self.summary}" if self.summary else "")
        return [{"role": "system", "content": sys}] + self.recent

    def say(self, user_text):
        self.recent.append({"role": "user", "content": user_text})
        r = client.chat.completions.create(model=MODEL, messages=self.messages(), temperature=0)
        reply = r.choices[0].message.content
        self.recent.append({"role": "assistant", "content": reply})
        # if recent grew too big, fold the oldest pair into the summary
        if len(self.recent) > self.keep * 2:
            old, self.recent = self.recent[:2], self.recent[2:]
            self.summary = self._summarize(old)
        return reply

m = SummaryMemory("You are a concise assistant.", keep_turns=2)
print(m.say("My favourite colour is teal."))
print(m.say("I have a dog named Pixel."))
print(m.say("I live in Bilbao."))
print(m.say("What colour did I mention, and what's my dog's name?"))   # recalled via the summary
print("\n[running summary]:", m.summary)

This time the early facts survive — not because we kept every message, but because we compressed them into the summary. Memory is a budgeting problem. You decide what's worth spending context tokens on.

::::{dropdown} 🔍 The two strategies, and where each fits
:color: info

```
Window   [system | ....... last N turns .......]        simple, forgets old facts
Summary  [system + summary | .. last N turns ..]        keeps old facts, costs a summarizer call
```

Real systems combine both, and add a third: **retrieval** — instead of summarizing, store old turns in a database and fetch only the relevant ones on demand. That's RAG, and it's the same trick applied to documents. You'll build it as the knowledge agent in **1.7**.

**On counting tokens:** to budget precisely you need to *count* tokens, not guess. Libraries like `tiktoken` do this; most of the time the `usage` field on each response (0.1) is enough to see the trend.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Tune the window.** Set `keep_turns=5` in `WindowMemory` and re-ask about the colour. Bigger window, more memory, more tokens per call — the whole trade-off in one number.
2. **Inspect the messages.** Print `m.messages()` after a few turns of `SummaryMemory` to see exactly what gets sent: system + summary + recent.
3. **Break the summarizer.** Change the summary instruction to "be extremely brief" and watch which facts survive and which get dropped. Summarization is lossy — you choose the losses.
::::

**Block 0 complete.** You've built, by hand: model calls, structured output, the agent loop, workflows, reflection, human-in-the-loop, and memory. You understand every moving part of an agent.

**What's next — Block 1:** we pick up **PydanticAI** and rebuild this on a clean, typed foundation. First stop, **1.0**: the same agent from 0.3, rewritten in PydanticAI — so you can see exactly what the framework does for you, and what it doesn't.